# Unitree Go2：PPOによる四足歩行学習と120秒評価

Genesis上で256体のUnitree Go2を並列シミュレーションし、PPOで12関節の位置指令を学習します。観測は46次元で、胴体のローカル速度・角速度・重力方向、目標前進速度、関節角度差・関節速度、直前の行動から構成します。学習後は方策を保存し、1体の評価環境で120秒間の歩行と速度追従を評価します。

実行順序は、保存先の作成、学習条件の設定、学習環境の構築、観測・報酬関数の定義、PPO学習、グラフ保存、評価環境の構築、120秒動画の生成です。


In [ ]:
# ============================================================
# CELL 1 — 事前準備：Drive接続・保存先作成・依存関係
# ============================================================
import os, sys, subprocess, datetime, json, math, random, shutil

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT_DIR = "/content/drive/MyDrive/Go2_RL"
else:
    ROOT_DIR = os.path.abspath("./Go2_RL")

RUN_ID = datetime.datetime.now().strftime("go2_v11_%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(ROOT_DIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

print("RUN_DIR =", RUN_DIR)

# Colabでは不足パッケージを準備
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "genesis-world", "imageio", "imageio-ffmpeg", "pandas", "matplotlib"],
                   check=True)

print("事前準備完了。以降の成果物は自動的にDriveへ保存します。")

In [ ]:
# ============================================================
# CELL 2 — import / seed / 全設定
# ============================================================
import os, math, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import imageio.v2 as imageio
import genesis as gs

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- 固定仕様 ----
NUM_ENVS = 256
NUM_OBS = 46
NUM_ACT = 12
ROLLOUT_STEPS = 64
PPO_EPOCHS = 4
MINIBATCH_SIZE = 4096
TOTAL_ITERS = 3000
CHECKPOINT_EVERY = 250
DECIMATION = 1

GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.2
LR = 3e-4
ENTROPY_COEF = 0.005
VALUE_COEF = 0.5
MAX_GRAD_NORM = 1.0

ACTION_SCALE = 0.25
KP = 20.0
KD = 0.5
MAX_EPISODE_STEPS = 1000

# Stage: (開始iteration, command最小, command最大)
# (段階, 目標速度下限, 上限)。習熟条件を満たした時のみ進級する。
CURRICULUM = [
    (0, 0.00, 0.00),
    (1, 0.05, 0.15),
    (2, 0.15, 0.30),
    (3, 0.30, 0.50),
    (4, 0.40, 0.70),
]
STAGE_MIN_ITERS = [100, 150, 200, 250, 0]
STAGE_MAX_ITERS = [600, 900, 1100, 1400, 10**9]
# stage0は立位維持、stage1以降は前進速度・姿勢・転倒率で進級。
STAGE_FALL_MAX = [0.02, 0.05, 0.07, 0.09, 1.0]
STAGE_VEL_ERROR_MAX = [0.12, 0.12, 0.15, 0.18, 1.0]
STAGE_ORIENT_MIN = [0.85, 0.80, 0.75, 0.70, 0.0]
STAGE_WINDOW = 40
# true=最大滞在iterationに達しても習熟しない場合は進級せず、その段階に留まる
REQUIRE_MASTERY = True

# Genesis初期化
try:
    gs.init(backend=gs.gpu, precision="32", seed=SEED)
except Exception:
    gs.init(backend=gs.cpu, precision="32", seed=SEED)

DEVICE = gs.device
print("Genesis device:", DEVICE)

CONFIG = {
    "VERSION": "11", "NUM_ENVS": NUM_ENVS, "NUM_OBS": NUM_OBS, "NUM_ACT": NUM_ACT,
    "ROLLOUT_STEPS": ROLLOUT_STEPS, "TOTAL_ITERS": TOTAL_ITERS,
    "ACTION_SCALE": ACTION_SCALE, "KP": KP, "KD": KD,
    "CURRICULUM": CURRICULUM, "STAGE_MIN_ITERS": STAGE_MIN_ITERS, "STAGE_MAX_ITERS": STAGE_MAX_ITERS
}
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)
print("設定保存:", os.path.join(RUN_DIR, "config.json"))

## 学習対象と段階学習

12関節の基準角度を中心に、Actorの12次元出力に`ACTION_SCALE=0.25`を乗じた補正を加えて関節位置を指令します。関節の位置制御ゲインは`KP=20.0`、速度制御ゲインは`KD=0.5`です。物理ステップは0.02秒、`DECIMATION=1`なので各制御指令に対して物理計算を1回実行します。

目標前進速度は5段階で、立位（0）、0.05～0.15、0.15～0.30、0.30～0.50、0.40～0.70 m/sです。直近40 iterationの転倒割合・速度誤差・姿勢報酬が各段階の基準を満たすと次段階に進みます。基準を満たさなければ現在の段階で学習を続けます。

報酬には速度追従、姿勢、高さ、横・上下速度、角速度、関節角度・速度、行動変化、脚末端リンクによる歩容と滑りの近似指標、転倒ペナルティを組み合わせます。


## 学習環境の構築

CELL 3ではGenesisのSceneに床面とGo2を追加し、256環境を構築します。関節名から12自由度のインデックスを取得してPDゲインと初期姿勢を設定します。Viewerの更新頻度は60に設定します。


## 脚末端リンクを用いた評価

Go2の脚末端リンクを取得し、その位置と速度から脚の挙上パターンと滑りの近似値を計算します。`foot`または`toe`に相当するリンクを優先し、見つからない場合は各脚の`calf`リンクを使用します。これらの量は報酬の補助成分として使用します。


In [ ]:
# ============================================================
# CELL 3 — 学習Scene / Go2 / 関節
# ============================================================
scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.02, substeps=2),
    viewer_options=gs.options.ViewerOptions(refresh_rate=60),
    show_viewer=False,
)
plane = scene.add_entity(gs.morphs.Plane())
robot = scene.add_entity(
    gs.morphs.URDF(
        file="urdf/go2/urdf/go2.urdf",
        pos=(0.0, 0.0, 0.42),
    )
)
scene.build(n_envs=NUM_ENVS, env_spacing=(1.5, 1.5))

JOINT_NAMES = [
    "FR_hip_joint","FR_thigh_joint","FR_calf_joint",
    "FL_hip_joint","FL_thigh_joint","FL_calf_joint",
    "RR_hip_joint","RR_thigh_joint","RR_calf_joint",
    "RL_hip_joint","RL_thigh_joint","RL_calf_joint",
]
# Genesis公式Go2例はjoint.dof_startを使用。古いAPIだけidx_localへフォールバック。
dof_idx = [getattr(robot.get_joint(n), "dof_start", getattr(robot.get_joint(n), "dof_idx_local", None)) for n in JOINT_NAMES]
assert all(x is not None for x in dof_idx), "関節DOF index取得に失敗"
assert len(dof_idx) == NUM_ACT, f"関節数={len(dof_idx)}; 12必要です"

# URDF順序に対応する安定した基準姿勢
default_dof_pos = torch.tensor([
    0.0, 0.80, -1.50,   # FR
    0.0, 0.80, -1.50,   # FL
    0.0, 1.00, -1.50,   # RR
    0.0, 1.00, -1.50,   # RL
], device=DEVICE, dtype=torch.float32)

assert default_dof_pos.shape == (NUM_ACT,)

robot.set_dofs_kp(torch.full((NUM_ACT,), KP, device=DEVICE), dofs_idx_local=dof_idx)
robot.set_dofs_kv(torch.full((NUM_ACT,), KD, device=DEVICE), dofs_idx_local=dof_idx)

# Genesis APIの実行前診断（最初の重い学習前）
assert tuple(robot.get_quat().shape) == (NUM_ENVS,4)
assert tuple(robot.get_vel().shape) == (NUM_ENVS,3)
assert tuple(robot.get_ang().shape) == (NUM_ENVS,3)
assert tuple(robot.get_dofs_position(dofs_idx_local=dof_idx).shape) == (NUM_ENVS,12)
assert tuple(robot.get_dofs_velocity(dofs_idx_local=dof_idx).shape) == (NUM_ENVS,12)
# 足先リンク名はURDFの実物から検出し、4脚揃わなければ学習開始前に停止。
link_names=[link.name for link in robot.links]
# Go2 URDFには独立した foot/toe link が無い版があります。
# その場合、各脚の末端剛体は *_calf です。
# まず foot/toe を探し、無ければ calf を末端リンクとして使用します。
FOOT_NAMES=[]
for leg in ["FR","FL","RR","RL"]:
    leg_links=[name for name in link_names if name.upper().startswith(leg+"_")]
    preferred=[
        name for name in leg_links
        if ("foot" in name.lower() or "toe" in name.lower())
    ]
    if len(preferred)==1:
        chosen=preferred[0]
    else:
        calf=[name for name in leg_links if name.lower()==f"{leg.lower()}_calf"]
        if len(calf)!=1:
            raise RuntimeError(
                f"{leg}の末端脚リンクを識別できません: "
                f"leg_links={leg_links}; 全リンク={link_names}"
            )
        chosen=calf[0]
    FOOT_NAMES.append(chosen)

feet_links=[robot.get_link(name=n) for n in FOOT_NAMES]
print("歩行評価に使用する末端脚リンク:", FOOT_NAMES)
assert len(feet_links)==4
for link in feet_links:
    assert tuple(link.get_pos().shape)==(NUM_ENVS,3)
    assert tuple(link.get_vel().shape)==(NUM_ENVS,3)
print("足先リンク検査OK:", FOOT_NAMES)
print("Scene / RigidEntity API shape check OK")
print("Scene OK:", NUM_ENVS, "envs")
print("dof_idx:", dof_idx)
print("default_dof_pos:", tuple(default_dof_pos.shape))

In [ ]:
# ============================================================
# CELL 4 — Quaternion / Observation / Reset / Reward
# ============================================================
def quat_conjugate(q):
    out = q.clone()
    out[..., 1:] *= -1
    return out

def quat_mul(q, r):
    w1,x1,y1,z1 = q.unbind(-1); w2,x2,y2,z2 = r.unbind(-1)
    return torch.stack((
        w1*w2-x1*x2-y1*y2-z1*z2,
        w1*x2+x1*w2+y1*z2-z1*y2,
        w1*y2-x1*z2+y1*w2+z1*x2,
        w1*z2+x1*y2-y1*x2+z1*w2
    ), dim=-1)

def quat_rotate_inverse(q, v):
    assert q.ndim == 2 and q.shape[1] == 4
    assert v.ndim == 2 and v.shape[1] == 3
    qv = torch.cat((torch.zeros((q.shape[0],1), device=q.device, dtype=v.dtype), v), dim=1)
    return quat_mul(quat_mul(quat_conjugate(q), qv), q)[...,1:]

def curriculum_command(stage, n):
    _, lo, hi = CURRICULUM[stage]
    if hi == lo:
        cmd=torch.full((n,1),lo,device=DEVICE)
    else:
        cmd=torch.empty((n,1),device=DEVICE).uniform_(lo,hi)
    return stage,cmd

def get_obs(robot_obj, last_action, command, batch):
    # 全てbatchを明示。学習256体と評価1体を同じ関数で安全に扱う。
    base_quat = robot_obj.get_quat()
    lin_world = robot_obj.get_vel()
    ang_world = robot_obj.get_ang()

    dof_pos = robot_obj.get_dofs_position(dofs_idx_local=dof_idx)
    dof_vel = robot_obj.get_dofs_velocity(dofs_idx_local=dof_idx)

    expected = {
        "quat": (batch,4), "lin": (batch,3), "ang": (batch,3),
        "dof_pos": (batch,12), "dof_vel": (batch,12),
        "last_action": (batch,12), "command": (batch,1)
    }
    actual = {
        "quat": base_quat, "lin": lin_world, "ang": ang_world,
        "dof_pos": dof_pos, "dof_vel": dof_vel,
        "last_action": last_action, "command": command
    }
    for name,x in actual.items():
        if tuple(x.shape) != expected[name]:
            raise RuntimeError(f"{name} shape={tuple(x.shape)}, expected={expected[name]}")

    lin_local = quat_rotate_inverse(base_quat, lin_world)
    ang_local = quat_rotate_inverse(base_quat, ang_world)
    gravity_world = torch.tensor([0.,0.,-1.],device=DEVICE).expand(batch,3)
    projected_gravity = quat_rotate_inverse(base_quat, gravity_world)
    dof_pos_err = dof_pos - default_dof_pos.unsqueeze(0)

    obs = torch.cat([
        lin_local,
        ang_local * 0.25,
        projected_gravity,
        command,
        dof_pos_err,
        dof_vel * 0.05,
        last_action
    ], dim=1)
    assert obs.shape == (batch, NUM_OBS), f"obs={obs.shape}, expected={(batch,NUM_OBS)}"
    if not torch.isfinite(obs).all():
        raise RuntimeError("観測obsにNaN/Infが発生しました。物理状態を確認してください")
    return torch.clamp(obs, -5.0, 5.0)

def compute_reward(robot_obj, action, prev_action, command, batch, stage, gait_phase):
    q=robot_obj.get_quat()
    pos=robot_obj.get_pos()
    lin_local=quat_rotate_inverse(q,robot_obj.get_vel())
    ang_local=quat_rotate_inverse(q,robot_obj.get_ang())
    dof_pos=robot_obj.get_dofs_position(dofs_idx_local=dof_idx)
    dof_vel=robot_obj.get_dofs_velocity(dofs_idx_local=dof_idx)
    gravity=quat_rotate_inverse(q,torch.tensor([0.,0.,-1.],device=DEVICE).expand(batch,3))
    vel_error=(lin_local[:,0]-command[:,0])
    r_vel=torch.exp(-(vel_error**2)/0.0625)
    r_orient=torch.exp(-torch.sum(gravity[:,:2]**2,dim=1)/0.08)
    r_height=torch.exp(-((pos[:,2]-0.30)**2)/0.01)
    p_lateral=lin_local[:,1]**2
    p_vertical=lin_local[:,2]**2
    p_ang=torch.sum(ang_local[:,:2]**2,dim=1)
    p_joint=torch.mean((dof_pos-default_dof_pos.unsqueeze(0))**2,dim=1)
    p_joint_vel=torch.mean(dof_vel**2,dim=1)
    p_action_rate=torch.mean((action-prev_action)**2,dim=1)

    # 足先高さは接地センサの代替ではなく「接地近傍」の近似。
    foot_pos=torch.stack([link.get_pos() for link in feet_links],dim=1)
    foot_vel=torch.stack([link.get_vel() for link in feet_links],dim=1)
    if foot_pos.shape!=(batch,4,3) or foot_vel.shape!=(batch,4,3):
        raise RuntimeError(f"足先shape異常 pos={foot_pos.shape}, vel={foot_vel.shape}")
    foot_z=foot_pos[:,:,2]
    near_ground=(foot_z<0.055).float()
    slip=(near_ground*(foot_vel[:,:,:2]**2).sum(-1)).mean(-1)
    # FR+RL / FL+RRを交互に上げるトロット参照。高さのみを弱く誘導し、
    # 強制的な関節軌道追従ではなく速度追従が主目的。
    phase=gait_phase[:,None]+torch.tensor([0.,math.pi,math.pi,0.],device=DEVICE)[None,:]
    desired_lift=(torch.sin(phase).clamp(min=0.0))*0.065
    gait_score=torch.exp(-((foot_z-desired_lift-0.02)**2).mean(-1)/0.006)
    air_fraction=(foot_z>0.065).float().mean(-1)
    moving=(command[:,0]>0.04).float()
    gait_weight=[0.0,0.12,0.20,0.24,0.20][stage]
    total=(
        2.5*r_vel+0.9*r_orient+0.5*r_height
        -0.25*p_lateral-0.20*p_vertical-0.08*p_ang
        -0.05*p_joint-0.0004*p_joint_vel-0.06*p_action_rate
        +moving*gait_weight*gait_score
        -moving*0.12*slip
    )
    fallen=(pos[:,2]<0.18)|(gravity[:,2]>-0.45)
    total=total+torch.where(fallen,-5.0,torch.zeros_like(total))
    if not torch.isfinite(total).all():
        raise RuntimeError("rewardにNaN/Infが発生しました")
    parts={
        "r_vel":r_vel.mean().item(),"r_orient":r_orient.mean().item(),
        "r_height":r_height.mean().item(),"p_action_rate":p_action_rate.mean().item(),
        "actual_vx":lin_local[:,0].mean().item(),
        "fall_frac_step":fallen.float().mean().item(),
        "velocity_error":vel_error.abs().mean().item(),
        "gait_score":gait_score.mean().item(),"foot_slip_proxy":slip.mean().item(),
        "air_fraction":air_fraction.mean().item(),
    }
    return total,fallen,parts

def reset_envs(env_ids, last_action, episode_len):
    if env_ids.numel() == 0:
        return
    n = env_ids.numel()
    pos = torch.zeros((n,3),device=DEVICE)
    pos[:,2] = 0.42
    quat = torch.zeros((n,4),device=DEVICE); quat[:,0]=1.0
    qpos = default_dof_pos.unsqueeze(0).repeat(n,1)
    robot.set_pos(pos, envs_idx=env_ids, zero_velocity=True)
    robot.set_quat(quat, envs_idx=env_ids, zero_velocity=True)
    robot.set_dofs_position(qpos, dofs_idx_local=dof_idx, envs_idx=env_ids, zero_velocity=True)
    last_action[env_ids] = 0.0
    episode_len[env_ids] = 0

print("関数定義OK")

def ensure_batch(x, width, name):
    """Normalize a single-environment Genesis output to (1,width)."""
    if x.ndim == 1 and x.shape[0] == width:
        x = x.unsqueeze(0)
    if tuple(x.shape) != (1,width):
        raise RuntimeError(f"{name}: {tuple(x.shape)} != (1,{width})")
    return x


In [ ]:
# ============================================================
# CELL 5 — Actor-Critic / PPO
# ============================================================
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(NUM_OBS,512), nn.ELU(),
            nn.Linear(512,256), nn.ELU(),
            nn.Linear(256,128), nn.ELU(),
            nn.Linear(128,NUM_ACT)
        )
        self.critic = nn.Sequential(
            nn.Linear(NUM_OBS,512), nn.ELU(),
            nn.Linear(512,256), nn.ELU(),
            nn.Linear(256,128), nn.ELU(),
            nn.Linear(128,1)
        )
        self.log_std = nn.Parameter(torch.full((NUM_ACT,), -1.2))

    def forward(self, obs):
        assert obs.ndim == 2 and obs.shape[1] == NUM_OBS
        if not torch.isfinite(obs).all():
            raise RuntimeError("ActorCritic入力obsにNaN/Infがあります")

        mu = self.actor(obs)
        value = self.critic(obs).squeeze(-1)

        # log_stdは学習パラメータなので、NaNを黙って置換せず異常なら即停止。
        if not torch.isfinite(self.log_std).all():
            raise RuntimeError(
                "log_stdにNaN/Infが発生しました。直前のPPO更新が数値発散しています。"
            )

        # exp()前に安全範囲へ制限。stdは常に正値。
        safe_log_std = torch.clamp(self.log_std, -5.0, 1.0)
        std = torch.exp(safe_log_std).expand_as(mu).clamp(1e-4, 3.0)

        if not torch.isfinite(mu).all():
            raise RuntimeError("Actor muにNaN/Infが発生しました")
        if not torch.isfinite(value).all():
            raise RuntimeError("Critic valueにNaN/Infが発生しました")
        if not torch.isfinite(std).all() or (std <= 0).any():
            raise RuntimeError("Actor stdが不正です")

        assert mu.shape == (obs.shape[0], NUM_ACT)
        assert std.shape == mu.shape
        assert value.shape == (obs.shape[0],)
        return mu, std, value

model = ActorCritic().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, eps=1e-5)

def evaluate_actions(obs, actions):
    mu, std, value = model(obs)
    if not torch.isfinite(actions).all():
        raise RuntimeError("evaluate_actions: actionsにNaN/Infがあります")
    dist = Normal(mu, std, validate_args=False)
    logp = dist.log_prob(actions).sum(-1)
    entropy = dist.entropy().sum(-1)
    if not torch.isfinite(logp).all():
        raise RuntimeError("PPO log_probにNaN/Infが発生しました")
    return logp, entropy, value

def assert_finite(name, x):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(
            f"{name} にNaN/Infがあります: {bad}/{x.numel()} elements"
        )

def assert_model_finite(model_obj, where="model"):
    for name, p in model_obj.named_parameters():
        if not torch.isfinite(p).all():
            raise RuntimeError(f"{where}: parameter {name} にNaN/Infがあります")

# dummy shape test before expensive training
dummy=torch.zeros((7,NUM_OBS),device=DEVICE)
mu,std,val=model(dummy)
assert mu.shape==(7,12) and std.shape==(7,12) and val.shape==(7,)
print("Network shape test OK:", mu.shape, val.shape)

## PPO学習と数値検査

Actor-Criticは46次元の観測から12次元の行動分布と状態価値を計算します。各iterationで256環境×64ステップの経験を収集し、GAEでAdvantageを計算して、4 epochのPPO更新を行います。PPO Clipは0.2、学習率は0.0003、割引率は0.99、GAE λは0.95です。

コードでは観測、行動分布、報酬、Advantage、損失、勾配などの数値を検査し、非有限値が検出された場合は例外を発生させます。勾配ノルムは最大1.0に制限します。250 iterationごとにチェックポイントと学習指標を保存し、終了時に学習済み方策を保存します。


In [ ]:
# ============================================================
# CELL 6 — 完全自動PPO学習
# ============================================================
B=NUM_ENVS
last_action=torch.zeros((B,NUM_ACT),device=DEVICE)
episode_len=torch.zeros(B,dtype=torch.long,device=DEVICE)
all_ids=torch.arange(B,device=DEVICE)
reset_envs(all_ids,last_action,episode_len)

metrics=[]
prev_stage=-1
current_stage=0
stage_age=0
recent_rows=[]
gait_phase=torch.zeros(B,device=DEVICE)
start_wall=time.time()
command=torch.zeros((B,1),device=DEVICE)

for it in range(TOTAL_ITERS):
    stage,new_command=curriculum_command(current_stage,B)
    # Stage切替時にcommandを更新。Stage内も少し多様化するため毎iteration再サンプル。
    command=new_command
    stage_age+=1
    if stage!=prev_stage:
        print(f"\n=== Stage {stage} 開始 / iteration {it} / command {CURRICULUM[stage][1:]} m/s ===")
        prev_stage=stage

    obs_buf=torch.zeros((ROLLOUT_STEPS,B,NUM_OBS),device=DEVICE)
    act_buf=torch.zeros((ROLLOUT_STEPS,B,NUM_ACT),device=DEVICE)
    logp_buf=torch.zeros((ROLLOUT_STEPS,B),device=DEVICE)
    rew_buf=torch.zeros((ROLLOUT_STEPS,B),device=DEVICE)
    val_buf=torch.zeros((ROLLOUT_STEPS,B),device=DEVICE)
    done_buf=torch.zeros((ROLLOUT_STEPS,B),dtype=torch.bool,device=DEVICE)

    part_acc={k:0.0 for k in ["r_vel","r_orient","r_height","p_action_rate",
                 "actual_vx","fall_frac_step","velocity_error","gait_score",
                 "foot_slip_proxy","air_fraction"]}

    for t in range(ROLLOUT_STEPS):
        obs=get_obs(robot,last_action,command,B)
        with torch.no_grad():
            mu,std,value=model(obs)
            assert_finite("rollout mu", mu)
            assert_finite("rollout std", std)
            dist=Normal(mu,std,validate_args=False)

            # PPOが記録するactionと実機(シミュレータ)へ送るactionを一致させる。
            # 極端な外れ値を避けるため、正規分布サンプルを[-1,1]へclamp。
            action=dist.sample()
            assert_finite("sampled action", action)
            logp=dist.log_prob(action).sum(-1)
            assert_finite("rollout logp", logp)

        env_action=torch.clamp(action,-1.0,1.0)
        target=default_dof_pos.unsqueeze(0)+env_action*ACTION_SCALE
        assert target.shape==(B,NUM_ACT)
        robot.control_dofs_position(target,dofs_idx_local=dof_idx)

        for _ in range(DECIMATION):
            scene.step()

        gait_phase=(gait_phase+2*math.pi*1.7*0.02)%(2*math.pi)
        reward,fallen,parts=compute_reward(robot,env_action,last_action,command,B,stage,gait_phase)
        assert_finite("reward", reward)
        assert_finite("value", value)
        episode_len += 1
        timeout=episode_len>=MAX_EPISODE_STEPS
        done=fallen|timeout

        obs_buf[t]=obs
        act_buf[t]=action
        logp_buf[t]=logp
        rew_buf[t]=reward
        val_buf[t]=value
        done_buf[t]=done

        for k in part_acc: part_acc[k]+=parts[k]

        last_action=env_action.detach().clone()
        reset_ids=torch.nonzero(done,as_tuple=False).squeeze(-1)
        reset_envs(reset_ids,last_action,episode_len)

    # Bootstrap
    with torch.no_grad():
        next_obs=get_obs(robot,last_action,command,B)
        _,_,next_value=model(next_obs)

    # GAE: done[t]を同じtransition tに対応させる
    adv=torch.zeros_like(rew_buf)
    gae=torch.zeros(B,device=DEVICE)
    for t in reversed(range(ROLLOUT_STEPS)):
        nonterminal=(~done_buf[t]).float()
        nv=next_value if t==ROLLOUT_STEPS-1 else val_buf[t+1]
        delta=rew_buf[t]+GAMMA*nv*nonterminal-val_buf[t]
        gae=delta+GAMMA*GAE_LAMBDA*nonterminal*gae
        adv[t]=gae
    returns=adv+val_buf

    flat_obs=obs_buf.reshape(-1,NUM_OBS)
    flat_act=act_buf.reshape(-1,NUM_ACT)
    flat_old_logp=logp_buf.reshape(-1)
    flat_adv=adv.reshape(-1)
    flat_ret=returns.reshape(-1)

    N=flat_obs.shape[0]
    assert N==ROLLOUT_STEPS*B
    assert_finite("flat_obs", flat_obs)
    assert_finite("flat_act", flat_act)
    assert_finite("flat_old_logp", flat_old_logp)
    assert_finite("flat_adv before normalize", flat_adv)
    assert_finite("flat_ret", flat_ret)

    adv_std=flat_adv.std(unbiased=False)
    if (not torch.isfinite(adv_std)) or adv_std < 1e-8:
        adv_std=torch.tensor(1.0,device=DEVICE)
    flat_adv=(flat_adv-flat_adv.mean())/(adv_std+1e-8)
    assert_finite("flat_adv normalized", flat_adv)

    last_policy_loss=last_value_loss=last_entropy=0.0
    for epoch in range(PPO_EPOCHS):
        perm=torch.randperm(N,device=DEVICE)
        for start in range(0,N,MINIBATCH_SIZE):
            idx=perm[start:start+MINIBATCH_SIZE]
            new_logp,entropy,new_value=evaluate_actions(flat_obs[idx],flat_act[idx])
            ratio=torch.exp(new_logp-flat_old_logp[idx])
            s1=ratio*flat_adv[idx]
            s2=torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*flat_adv[idx]
            policy_loss=-torch.min(s1,s2).mean()
            value_loss=0.5*((new_value-flat_ret[idx])**2).mean()
            loss=policy_loss+VALUE_COEF*value_loss-ENTROPY_COEF*entropy.mean()

            # NaN/Infのminibatchでモデルを破壊しない
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"PPO lossがNaN/Infです: iteration={it}, epoch={epoch}, start={start}"
                )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            # 勾配自体を検査してから更新
            for pname,p in model.named_parameters():
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    raise RuntimeError(
                        f"gradient {pname} にNaN/Inf: iteration={it}, epoch={epoch}"
                    )

            grad_norm=nn.utils.clip_grad_norm_(
                model.parameters(),MAX_GRAD_NORM,error_if_nonfinite=True
            )
            optimizer.step()

            # log_stdは更新後も明示的に安全範囲へ戻す
            with torch.no_grad():
                model.log_std.clamp_(-5.0,1.0)

            assert_model_finite(model,where=f"after PPO update it={it}")
            last_policy_loss=policy_loss.item()
            last_value_loss=value_loss.item()
            last_entropy=entropy.mean().item()

    row={
        "iteration":it,"stage":stage,
        "mean_reward":rew_buf.mean().item(),
        "target_vx":command.mean().item(),
        "actual_vx":part_acc["actual_vx"]/ROLLOUT_STEPS,
        "r_vel":part_acc["r_vel"]/ROLLOUT_STEPS,
        "r_orient":part_acc["r_orient"]/ROLLOUT_STEPS,
        "r_height":part_acc["r_height"]/ROLLOUT_STEPS,
        "action_rate":part_acc["p_action_rate"]/ROLLOUT_STEPS,
        "fall_frac":part_acc["fall_frac_step"]/ROLLOUT_STEPS,
        "policy_loss":last_policy_loss,"value_loss":last_value_loss,
        "entropy":last_entropy,
        "log_std_mean":model.log_std.detach().mean().item(),
        "velocity_error":part_acc["velocity_error"]/ROLLOUT_STEPS,
        "gait_score":part_acc["gait_score"]/ROLLOUT_STEPS,
        "foot_slip_proxy":part_acc["foot_slip_proxy"]/ROLLOUT_STEPS,
        "air_fraction":part_acc["air_fraction"]/ROLLOUT_STEPS
    }
    metrics.append(row)
    recent_rows.append(row)
    if len(recent_rows)>STAGE_WINDOW:
        recent_rows.pop(0)
    if stage<4 and stage_age>=STAGE_MIN_ITERS[stage] and len(recent_rows)>=STAGE_WINDOW:
        mean=lambda key: sum(r[key] for r in recent_rows)/len(recent_rows)
        mastered=(mean("fall_frac")<STAGE_FALL_MAX[stage]
                  and mean("velocity_error")<STAGE_VEL_ERROR_MAX[stage]
                  and mean("r_orient")>STAGE_ORIENT_MIN[stage])
        if mastered:
            print(f"★ Stage {stage} 習熟判定合格 → Stage {stage+1}: "
                  f"fall={mean('fall_frac'):.3f}, vel_error={mean('velocity_error'):.3f}")
            current_stage+=1
            stage_age=0
            recent_rows=[]
        elif stage_age>=STAGE_MAX_ITERS[stage] and stage_age%100==0:
            print(f"Stage {stage} 未習熟のため維持（経過{stage_age}iter）。学習を継続します")

    if it%25==0:
        print(f"it={it:4d} S={stage} R={row['mean_reward']:.3f} "
              f"vx={row['actual_vx']:.3f}/{row['target_vx']:.3f} "
              f"fall={row['fall_frac']:.3f} orient={row['r_orient']:.3f} "
              f"gait={row['gait_score']:.3f} slip={row['foot_slip_proxy']:.3f} "
              f"ETA={(time.time()-start_wall)/(it+1)*(TOTAL_ITERS-it-1)/3600:.1f}h")

    if (it+1)%CHECKPOINT_EVERY==0:
        ckpt=os.path.join(RUN_DIR,f"checkpoint_{it+1:04d}.pt")
        torch.save({"iteration":it+1,"model":model.state_dict(),
                    "optimizer":optimizer.state_dict(),"config":CONFIG,"stage":current_stage},ckpt)
        pd.DataFrame(metrics).to_csv(os.path.join(RUN_DIR,"training_metrics.csv"),index=False)

# 最終保存
POLICY_PATH=os.path.join(RUN_DIR,"go2_walking_policy_v11.pt")
torch.save(model.state_dict(),POLICY_PATH)
pd.DataFrame(metrics).to_csv(os.path.join(RUN_DIR,"training_metrics.csv"),index=False)
print("学習完了:", POLICY_PATH)

In [ ]:
# ============================================================
# CELL 7 — 学習曲線を自動保存
# ============================================================
df=pd.DataFrame(metrics)
plots=[
    ("mean_reward","Mean reward"),
    ("actual_vx","Actual forward velocity"),
    ("fall_frac","Fall fraction"),
    ("r_orient","Orientation reward"),
]
for col,title in plots:
    plt.figure(figsize=(10,4))
    plt.plot(df["iteration"],df[col])
    if col=="actual_vx":
        plt.plot(df["iteration"],df["target_vx"],label="target")
        plt.legend()
    plt.xlabel("iteration"); plt.ylabel(col); plt.title(title)
    plt.grid(alpha=0.25); plt.tight_layout()
    path=os.path.join(RUN_DIR,f"{col}.png")
    plt.savefig(path,dpi=160)
    plt.close()
    print("saved:",path)

In [ ]:
# ============================================================
# CELL 8 — 評価Sceneを1体専用で作成（学習batchを絶対に再利用しない）
# ============================================================
EVAL_BATCH=1
EVAL_TARGET_VX=0.45

eval_scene=gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.02,substeps=2),
    viewer_options=gs.options.ViewerOptions(refresh_rate=60),
    show_viewer=False,
)
eval_plane=eval_scene.add_entity(gs.morphs.Plane())
eval_robot=eval_scene.add_entity(
    gs.morphs.URDF(file="urdf/go2/urdf/go2.urdf",pos=(0,0,0.42))
)
camera=eval_scene.add_camera(
    res=(960,540), pos=(2.8,-3.0,1.8), lookat=(0,0,0.35), fov=45, GUI=False
)
# Single-environment evaluation: Genesis may squeeze leading batch dimension.
# Use n_envs=1 to request batched outputs where supported.
eval_scene.build(n_envs=EVAL_BATCH)

eval_dof_idx=[getattr(eval_robot.get_joint(n),"dof_start",getattr(eval_robot.get_joint(n),"dof_idx_local",None)) for n in JOINT_NAMES]
assert len(eval_dof_idx)==12
eval_robot.set_dofs_kp(torch.full((12,),KP,device=DEVICE),dofs_idx_local=eval_dof_idx)
eval_robot.set_dofs_kv(torch.full((12,),KD,device=DEVICE),dofs_idx_local=eval_dof_idx)
eval_robot.set_dofs_position(default_dof_pos.unsqueeze(0),dofs_idx_local=eval_dof_idx,zero_velocity=True)
# Ensure the base is reset explicitly (not only the 12 actuated joints).
eval_robot.set_pos(torch.tensor([[0.,0.,0.42]],device=DEVICE),zero_velocity=True)
eval_robot.set_quat(torch.tensor([[1.,0.,0.,0.]],device=DEVICE),zero_velocity=True)

# Camera preflight: fail with a useful message before rendering thousands of frames.
try:
    eval_scene.step()
    _rgb, _, _, _ = camera.render(rgb=True)
    assert _rgb is not None and np.asarray(_rgb).ndim == 3
    print("Camera preflight OK:", np.asarray(_rgb).shape)
except Exception as e:
    raise RuntimeError("カメラ初期化/描画に失敗しました。Genesisレンダラの設定を確認してください。") from e

eval_model=ActorCritic().to(DEVICE)
state=torch.load(POLICY_PATH,map_location=DEVICE)
eval_model.load_state_dict(state)
eval_model.eval()

eval_last_action=torch.zeros((1,12),device=DEVICE)
eval_command=torch.full((1,1),EVAL_TARGET_VX,device=DEVICE)

# 評価専用obs。eval_dof_idxを使うため明示的に分離。
def get_eval_obs():
    q=ensure_batch(eval_robot.get_quat(),4,"eval quat")
    lin=ensure_batch(eval_robot.get_vel(),3,"eval linear velocity")
    ang=ensure_batch(eval_robot.get_ang(),3,"eval angular velocity")
    qp=ensure_batch(eval_robot.get_dofs_position(dofs_idx_local=eval_dof_idx),12,"eval dof position")
    qv=ensure_batch(eval_robot.get_dofs_velocity(dofs_idx_local=eval_dof_idx),12,"eval dof velocity")
    tensors={"q":(q,(1,4)),"lin":(lin,(1,3)),"ang":(ang,(1,3)),
             "qp":(qp,(1,12)),"qv":(qv,(1,12)),
             "last":(eval_last_action,(1,12)),"cmd":(eval_command,(1,1))}
    for name,(x,shape) in tensors.items():
        if tuple(x.shape)!=shape:
            raise RuntimeError(f"EVAL {name}: {tuple(x.shape)} != {shape}")
    ll=quat_rotate_inverse(q,lin)
    aa=quat_rotate_inverse(q,ang)
    grav=quat_rotate_inverse(q,torch.tensor([0.,0.,-1.],device=DEVICE).expand(1,3))
    obs=torch.cat([ll,aa*0.25,grav,eval_command,
                   qp-default_dof_pos.unsqueeze(0),qv*0.05,eval_last_action],dim=1)
    assert obs.shape==(1,46),f"eval obs={obs.shape}"
    return torch.clamp(torch.nan_to_num(obs),-5,5)

test_obs=get_eval_obs()
with torch.no_grad():
    test_mu,_,_=eval_model(test_obs)
assert test_mu.shape==(1,12)
print("評価Preflight OK: obs",test_obs.shape,"action",test_mu.shape)

## 120秒の歩行評価

保存済みのActor-Critic方策を1体のGo2に読み込み、行動分布の平均値を用いて関節位置を決定します。評価は0.02秒刻みで120秒間行います。目標速度は0～20秒が0.20 m/s、20～50秒が0.40 m/s、50～80秒が0.60 m/s、80～110秒が0.40 m/s、110～120秒は0.40から0.10 m/sまで線形に減速します。

各ステップでロボット座標系の前進速度を取得し、目標速度との差を記録します。胴体の高さが0.18 m未満になると転倒回数を増やし、ロボットを初期姿勢に戻して評価を続けます。カメラはロボットの位置に追従し、映像を30 fps・120秒のMP4に保存します。終了時には転倒回数と平均絶対速度誤差を表示します。


In [ ]:
# CELL 9 — 120秒歩行評価・30fps動画生成
import os, imageio.v2 as imageio
import numpy as np
import torch

VIDEO_SECONDS=120
VIDEO_FPS=30
CONTROL_DT=0.02
TOTAL_CONTROL_STEPS=int(VIDEO_SECONDS/CONTROL_DT)
TOTAL_VIDEO_FRAMES=VIDEO_SECONDS*VIDEO_FPS
VIDEO_PATH=os.path.join(RUN_DIR,"go2_walking_120s_30fps.mp4")

def eval_command_for_time(sec):
    if sec < 20: return 0.20
    if sec < 50: return 0.40
    if sec < 80: return 0.60
    if sec < 110: return 0.40
    alpha=(sec-110.0)/10.0
    return 0.40*(1.0-alpha)+0.10*alpha

eval_robot.set_pos(torch.tensor([[0.,0.,0.42]],device=DEVICE),zero_velocity=True)
eval_robot.set_quat(torch.tensor([[1.,0.,0.,0.]],device=DEVICE),zero_velocity=True)
eval_robot.set_dofs_position(default_dof_pos.unsqueeze(0),dofs_idx_local=eval_dof_idx,zero_velocity=True)

last_action_eval=torch.zeros((1,NUM_ACT),device=DEVICE)
frames_written=0
vx_history=[]
cmd_history=[]
fall_events=0

writer=imageio.get_writer(VIDEO_PATH,fps=VIDEO_FPS,codec="libx264",quality=7,macro_block_size=None)
try:
    for step in range(TOTAL_CONTROL_STEPS):
        sec=step*CONTROL_DT
        cmd_value=eval_command_for_time(sec)
        command_eval=torch.tensor([[cmd_value]],device=DEVICE,dtype=torch.float32)

        eval_last_action=last_action_eval
        eval_command=command_eval
        obs_eval=get_eval_obs()
        if obs_eval.shape!=(1,NUM_OBS):
            raise RuntimeError(f"2分評価 obs shape異常: {tuple(obs_eval.shape)}")
        if not torch.isfinite(obs_eval).all():
            raise RuntimeError(f"2分評価 obsにNaN/Inf: t={sec:.2f}s")

        with torch.no_grad():
            mu,std,value=eval_model(obs_eval)
            action_eval=torch.clamp(mu,-1.,1.)
        if not torch.isfinite(action_eval).all():
            raise RuntimeError(f"2分評価 actionにNaN/Inf: t={sec:.2f}s")

        target_pos=default_dof_pos.unsqueeze(0)+ACTION_SCALE*action_eval
        eval_robot.control_dofs_position(target_pos,dofs_idx_local=eval_dof_idx)
        for _ in range(DECIMATION):
            eval_scene.step()

        pos_now=eval_robot.get_pos()
        vel_now=eval_robot.get_vel()
        if pos_now.ndim==1: pos_now=pos_now.unsqueeze(0)
        if vel_now.ndim==1: vel_now=vel_now.unsqueeze(0)
        quat_now=ensure_batch(eval_robot.get_quat(),4,"eval quat for speed")
        vel_local=quat_rotate_inverse(quat_now,ensure_batch(vel_now,3,"eval velocity for speed"))
        vx=float(vel_local[0,0].detach().cpu())
        vx_history.append(vx); cmd_history.append(cmd_value)

        if float(pos_now[0,2].detach().cpu()) < 0.18:
            fall_events+=1
            x=float(pos_now[0,0].detach().cpu())
            eval_robot.set_pos(torch.tensor([[x,0.,0.42]],device=DEVICE),zero_velocity=True)
            eval_robot.set_quat(torch.tensor([[1.,0.,0.,0.]],device=DEVICE),zero_velocity=True)
            eval_robot.set_dofs_position(default_dof_pos.unsqueeze(0),dofs_idx_local=eval_dof_idx,zero_velocity=True)
            last_action_eval=torch.zeros_like(last_action_eval)
        else:
            last_action_eval=action_eval.detach()

        frames_due=((step+1)*TOTAL_VIDEO_FRAMES)//TOTAL_CONTROL_STEPS - (step*TOTAL_VIDEO_FRAMES)//TOTAL_CONTROL_STEPS
        if frames_due:
            p=eval_robot.get_pos()
            if p.ndim>1: p=p[0]
            px,py,pz=[float(v.detach().cpu()) for v in p]
            camera.set_pose(pos=(px+2.2,py-2.2,max(1.0,pz+0.8)),lookat=(px,py,pz))
            rendered=camera.render(rgb=True)
            frame=rendered[0] if isinstance(rendered,(tuple,list)) else rendered
            if torch.is_tensor(frame): frame=frame.detach().cpu().numpy()
            frame=np.asarray(frame)
            if frame.dtype!=np.uint8:
                frame=((np.clip(frame,0,1)*255) if frame.max()<=1.0 else np.clip(frame,0,255)).astype(np.uint8)
            for _ in range(frames_due):
                writer.append_data(frame)
                frames_written+=1

        if step % int(10.0/CONTROL_DT)==0:
            print(f"評価 {sec:5.1f}/{VIDEO_SECONDS}s | cmd={cmd_value:.2f} | vx={vx:.2f} | 転倒={fall_events}")
finally:
    writer.close()

assert frames_written==TOTAL_VIDEO_FRAMES, f"frames={frames_written}, expected={TOTAL_VIDEO_FRAMES}"
mean_abs_error=float(np.mean(np.abs(np.asarray(vx_history)-np.asarray(cmd_history))))
print("\n=== 2分歩行評価 完了 ===")
print("動画:",VIDEO_PATH)
print("生成フレーム:",frames_written," 転倒回数:",fall_events)
print(f"平均速度追従誤差: {mean_abs_error:.3f} m/s")
try:
    from IPython.display import Video,display
    display(Video(VIDEO_PATH,embed=False,width=900))
except Exception as e:
    print("Notebook内動画表示は省略:",e)


## 生成されるファイルと評価指標

CELL 1で作成した`RUN_DIR`に、`config.json`、250 iterationごとの`checkpoint_*.pt`、学習推移の`training_metrics.csv`、学習済み方策`go2_walking_policy_v11.pt`を保存します。CELL 7では`mean_reward.png`、`actual_vx.png`、`fall_frac.png`、`r_orient.png`を生成します。CELL 9では`go2_walking_120s_30fps.mp4`を生成します。

学習曲線では報酬、実際の前進速度と目標速度、転倒割合、姿勢報酬を表示します。120秒評価ではロボット座標系の前進速度と目標速度から平均絶対速度誤差を計算し、胴体高さによる転倒イベント回数とともに表示します。
